# HAM10000 Skin Lesion Classification - ResNet50 Training

สมุดงานสำหรับฝึกฝนโมเดล ResNet-50 บนชุดข้อมูล HAM10000 (7 คลาส)
- ปรับแต่ง Hyperparameters ได้ที่ Cell ที่ 1
- รองรับ Automatic Mixed Precision (bfloat16) บน GPU RTX 6000
- แก้ปัญหา Class Imbalance ด้วย Balanced Class Weights
- บันทึกโมเดลที่ดีที่สุดลงใน `checkpoints/resnet50_best.pt`

In [ ]:
# =========================================================================
# 1. ปรับแต่ง Hyperparameters ที่นี่
# =========================================================================
CONFIG = {
    'model_name': 'resnet50',        # ชื่อโมเดลใน registry
    'img_size': 224,                 # ขนาดภาพ (224 x 224 สำหรับ ResNet-50)
    'batch_size': 64,                # ขนาด Batch
    'lr': 1e-4,                      # Learning Rate
    'weight_decay': 1e-4,            # L2 Regularization
    'epochs': 15,                    # จำนวนรอบเทรน
    'seed': 42,                      # Random Seed เพื่อความแม่นยำ
    'num_workers': 4,                # จำนวน Worker สำหรับ DataLoader
    'use_amp': True,                 # เปิดใช้ Mixed Precision (bf16)
    'use_class_weights': True,       # ใช้น้ำหนักคลาสแก้ Class Imbalance
    'use_weighted_sampler': False,   # สุ่ม batch ตามสัดส่วนคลาส
    'cache_in_ram': False,           # แคชภาพ uint8 ลง RAM
    'checkpoint_dir': 'checkpoints', # โฟลเดอร์บันทึกโมเดล
    'splits_file': 'splits/splits.csv',
    'data_dir': 'data',
}
print('Config loaded:', CONFIG)

In [ ]:
# =========================================================================
# 2. ตรวจสอบสเปก GPU และล็อก Seed
# =========================================================================
import os
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, f1_score

from src.utils import print_env_info, set_seed, get_amp_context
from src.augment import get_train_transform, get_val_transform
from src.dataset import HAM10000Dataset, CLASSES, get_class_weights, get_weighted_sampler
from src.models import get_model

set_seed(CONFIG['seed'])
print_env_info()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
Path(CONFIG['checkpoint_dir']).mkdir(parents=True, exist_ok=True)

In [ ]:
# =========================================================================
# 3. สร้าง Dataset และ DataLoader (Train / Val)
# =========================================================================
train_transform = get_train_transform(img_size=CONFIG['img_size'])
val_transform = get_val_transform(img_size=CONFIG['img_size'])

train_ds = HAM10000Dataset(
    splits_csv=CONFIG['splits_file'],
    split='train',
    data_dir=CONFIG['data_dir'],
    transform=train_transform,
    cache_in_ram=CONFIG['cache_in_ram'],
    img_size=CONFIG['img_size'],
)

val_ds = HAM10000Dataset(
    splits_csv=CONFIG['splits_file'],
    split='val',
    data_dir=CONFIG['data_dir'],
    transform=val_transform,
    cache_in_ram=CONFIG['cache_in_ram'],
    img_size=CONFIG['img_size'],
)

sampler = get_weighted_sampler(train_ds, split='train') if CONFIG['use_weighted_sampler'] else None
shuffle = (sampler is None)

train_loader = DataLoader(
    train_ds,
    batch_size=CONFIG['batch_size'],
    shuffle=shuffle,
    sampler=sampler,
    num_workers=CONFIG['num_workers'],
    pin_memory=True,
    persistent_workers=(CONFIG['num_workers'] > 0),
)

val_loader = DataLoader(
    val_ds,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=CONFIG['num_workers'],
    pin_memory=True,
    persistent_workers=(CONFIG['num_workers'] > 0),
)

print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")

In [ ]:
# =========================================================================
# 4. โหลดโมเดล ResNet-50, Loss Function, Optimizer
# =========================================================================
model = get_model(CONFIG['model_name'], num_classes=len(CLASSES), pretrained=True).to(device)

if CONFIG['use_class_weights']:
    class_weights = get_class_weights(CONFIG['splits_file'], split='train').to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    print('Using weighted CrossEntropyLoss:', [round(w.item(), 3) for w in class_weights])
else:
    criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'])

print('Model and optimizer ready on', device)

In [ ]:
# =========================================================================
# 5. Training & Validation Loop
# =========================================================================
best_val_f1 = 0.0
model_name = CONFIG['model_name']
best_ckpt_path = Path(CONFIG['checkpoint_dir']) / f"{model_name}_best.pt"

for epoch in range(1, CONFIG['epochs'] + 1):
    model.train()
    train_loss, train_correct, total_train = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with get_amp_context(use_amp=CONFIG['use_amp']):
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(labels)
        train_correct += (outputs.argmax(dim=1) == labels).sum().item()
        total_train += len(labels)
    scheduler.step()

    model.eval()
    val_loss, val_preds, val_targets = 0.0, [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with get_amp_context(use_amp=CONFIG['use_amp']):
                outputs = model(imgs)
                loss = criterion(outputs, labels)
            val_loss += loss.item() * len(labels)
            val_preds.extend(outputs.argmax(dim=1).cpu().tolist())
            val_targets.extend(labels.cpu().tolist())

    train_acc = train_correct / total_train
    val_acc = (torch.tensor(val_preds) == torch.tensor(val_targets)).float().mean().item()
    val_f1 = f1_score(val_targets, val_preds, average='macro')

    total_epochs = CONFIG['epochs']
    print(f"Epoch {epoch:02d}/{total_epochs:02d} | Train Loss: {train_loss/total_train:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss/len(val_ds):.4f} Acc: {val_acc:.4f} Macro-F1: {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save({
            'epoch': epoch,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'val_f1': val_f1,
            'config': CONFIG,
        }, best_ckpt_path)
        print(f"  >> Saved new best checkpoint to {best_ckpt_path} (F1: {best_val_f1:.4f})")

print('Training complete. Best Val Macro-F1:', round(best_val_f1, 4))

In [ ]:
# =========================================================================
# 6. ประเมินผลบน Test Set (Held-out Test)
# =========================================================================
print(f"Loading best checkpoint: {best_ckpt_path}")
ckpt = torch.load(best_ckpt_path, map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()

test_ds = HAM10000Dataset(
    splits_csv=CONFIG['splits_file'],
    split='test',
    data_dir=CONFIG['data_dir'],
    transform=val_transform,
)
test_loader = DataLoader(test_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'])

test_preds, test_targets = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device, non_blocking=True)
        with get_amp_context(use_amp=CONFIG['use_amp']):
            outputs = model(imgs)
        test_preds.extend(outputs.argmax(dim=1).cpu().tolist())
        test_targets.extend(labels.tolist())

print('\nClassification Report on TEST SET:')
print(classification_report(test_targets, test_preds, target_names=CLASSES, digits=4))